<a href="https://colab.research.google.com/github/JulianBotello01/PROGCOM-B/blob/main/NQ2_segund__Corte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from microbit import *
import music
import speaker

# ---------- CONFIGURACIÓN ----------
PIN_TIRA = pin0                 # Pin que controla el MOSFET de la tira LED
INTERVALO_MS = 50               # Tiempo entre mediciones (ms)
TIEMPO_SIN_MOVIMIENTO = 30_000  # 30 segundos (ms)
UMBRAL_MOVIMIENTO = 200         # Sensibilidad (menor = más sensible)

speaker.on()  # Activar altavoz interno

# ---------- FUNCIONES DE SONIDO ----------
def beep(freq=880, dur_ms=120):
    """Genera un beep corto en el altavoz interno."""
    music.pitch(freq, dur_ms)

def sonido_encendido():
    """Secuencia ascendente: luz encendida."""
    beep(784, 90); sleep(20)
    beep(988, 110); sleep(20)
    beep(1319, 120)

def sonido_apagado():
    """Secuencia descendente: luz apagada."""
    beep(988, 90); sleep(20)
    beep(784, 110); sleep(20)
    beep(523, 140)

# ---------- FUNCIONES DE MOVIMIENTO ----------
def encender_tira(estado):
    """Enciende o apaga la tira LED."""
    PIN_TIRA.write_digital(1 if estado else 0)

def leer_aceleracion():
    """Devuelve lecturas del acelerómetro."""
    return accelerometer.get_x(), accelerometer.get_y(), accelerometer.get_z()

def hay_movimiento(previos):
    """Evalúa si hubo movimiento."""
    x, y, z = leer_aceleracion()
    px, py, pz = previos
    cambio = abs(x - px) + abs(y - py) + abs(z - pz)
    sacudida = accelerometer.was_gesture('shake')
    movimiento = sacudida or (cambio > UMBRAL_MOVIMIENTO)
    return movimiento, (x, y, z)

# ---------- PROGRAMA PRINCIPAL ----------
display.off()  # Apagar LEDs para ahorrar batería

valores_previos = leer_aceleracion()
tira_encendida = False
ultimo_movimiento = 0

while True:
    movimiento, valores_previos = hay_movimiento(valores_previos)

    if not tira_encendida:
        # Si está apagada y se detecta movimiento → encender
        if movimiento:
            encender_tira(True)
            sonido_encendido()
            tira_encendida = True
            ultimo_movimiento = running_time()
    else:
        # Si está encendida y hay movimiento → reiniciar temporizador
        if movimiento:
            ultimo_movimiento = running_time()
        else:
            # Si no hay movimiento por 30 s → apagar
            if running_time() - ultimo_movimiento >= TIEMPO_SIN_MOVIMIENTO:
                encender_tira(False)
                sonido_apagado()
                tira_encendida = False
                ultimo_movimiento = 0

    sleep(INTERVALO_MS)